# Slovak AI Olympiad — Task 3 (Part 1): Wikipedia Corpus Extraction & Dataset Engineering
**Author:** Kristian Hudák  
**Competition:** National Olympiad in Artificial Intelligence (OAI) — Home Qualifier Round

This notebook builds the parallel sentence dataset from raw Slovak Wikipedia XML dumps:
1. Ingesting and streaming the Wikipedia XML dump
2. Sanitizing MediaWiki markup, templates, tables, HTML tags, and bracket citations via regular expressions
3. Filtering high-quality Slovak sentences based on length and linguistic punctuation constraints
4. Generating parallel pairs: original text with diacritics vs sanitized text without diacritics (`unicodedata.normalize`)
5. Calculating corpus descriptive statistics (`stats.json`) and exporting the curated dataset (`vety_dataset.xlsx`)


In [1]:
# 1. Import necessary libraries
import torch
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch.nn as nn
import json
import re
import unicodedata
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from google.colab import files

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Používané zariadenie: {device}")


Používané zariadenie: cpu


In [2]:
# 2. MediaWiki markup sanitizer, regex filters, and sentence validation
def vycisti_riadne(text):
    if not text: return ""
    # Remove MediaWiki templates {{...}} and tables {|...|}
    text = re.sub(r'\{\{.*?\}\}', '', text, flags=re.DOTALL)
    text = re.sub(r'\{\|.*?\|\}', '', text, flags=re.DOTALL)
    # Odstránenie súborov a kategórií
    text = re.sub(r'\[\[(?:Súbor|Obrázok|Kategória|File|Image):.*?\]\]', '', text, flags=re.IGNORECASE)
    # Odkazy [[článok|text]] -> text
    text = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]+)\]\]', r'\1', text)
    # HTML a formátovanie
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'={2,}.*?={2,}', '', text)
    text = re.sub(r"''+", '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def je_to_dobra_veta(veta):
    if not (50 <= len(veta) <= 2000):
        return False
    zakazane = ['{', '}', '|', '[', ']', '$', 'style=', 'class=', 'thumb', 'px', 'http']
    if any(z in veta for z in zakazane): return False
    slova = veta.split()
    if len(slova) < 5 or len(slova) > 50: return False
    if not veta[0].isupper(): return False
    return True


In [3]:
# 3. Stream XML dump and extract curated sentence samples
path = '/content/skwiki-20251120-pages-articles.xml'
vsetky_vety = []
CIEL_POCET = 20000

print(f"Spracovávam dokument, cieľ je {CIEL_POCET} viet...")

with open(path, 'r', encoding='utf-8') as f:
    buffer = ""
    for line in f:
        buffer += line
        if '</page>' in line:
            soup = BeautifulSoup(buffer, 'lxml-xml')
            ns = soup.find('ns')
            if ns and ns.get_text() == '0':
                text_tag = soup.find('text')
                if text_tag and text_tag.get_text():
                    cisty_obsah = vycisti_riadne(text_tag.get_text())
                    vety = re.split(r'(?<=[.!?])\s+(?=[A-ZÁÄČĎÉÍĹĽŇÓÔŔŠŤÚÝŽ])', cisty_obsah)
                    for veta in vety:
                        veta = veta.strip()
                        if je_to_dobra_veta(veta):
                            vsetky_vety.append(veta)
                        if len(vsetky_vety) >= CIEL_POCET:
                            break
            buffer = ""
            if len(vsetky_vety) >= CIEL_POCET:
                break
            if len(vsetky_vety) % 1000 == 0 and len(vsetky_vety) > 0:
                print(f"Nájdených {len(vsetky_vety)} viet...")


Spracovávam dokument, cieľ je 20000 viet...
Nájdených 15000 viet...


In [4]:
# 4. Build Pandas DataFrame and generate parallel diacritic-free inputs
def odstran_diakritiku(text):
    mapa = {
        'á': 'a', 'Á': 'A', 'ä': 'a', 'Ä': 'A', 'č': 'c', 'Č': 'C',
        'ď': 'd', 'Ď': 'D', 'é': 'e', 'É': 'E', 'í': 'i', 'Í': 'I',
        'ĺ': 'l', 'Ĺ': 'L', 'ľ': 'l', 'Ľ': 'L', 'ň': 'n', 'Ň': 'N',
        'ó': 'o', 'Ó': 'O', 'ô': 'o', 'Ô': 'O', 'ŕ': 'r', 'Ŕ': 'R',
        'š': 's', 'Š': 'S', 'ť': 't', 'Ť': 'T', 'ú': 'u', 'Ú': 'U',
        'ý': 'y', 'Ý': 'Y', 'ž': 'z', 'Ž': 'Z'
    }
    return "".join(mapa.get(c, c) for c in text)

df = pd.DataFrame(vsetky_vety, columns=['veta_s_diakritikou'])
df['veta_bez_diakritiky'] = df['veta_s_diakritikou'].apply(odstran_diakritiku)

# Rozdelenie na Train a Val (90/10)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"Dáta rozdelené: Train ({len(train_df)}), Val ({len(val_df)})")
df.head(10)


Dáta rozdelené: Train (18000), Val (2000)


,veta_s_diakritikou,veta_bez_diakritiky
0,Dary sú primárne určené na nákup vybavenia ser...,Dary su primarne urcene na nakup vybavenia ser...
1,Esperanto (pôvodne Lingvo Internacia – „medzin...,Esperanto (povodne Lingvo Internacia – „medzin...
2,"Názov je odvodený od pseudonymu, pod ktorým v ...","Nazov je odvodeny od pseudonymu, pod ktorym v ..."
3,Zámerom tvorcu bolo vytvoriť ľahko naučiteľný ...,Zamerom tvorcu bolo vytvorit lahko naucitelny ...
4,"Cieľom nebolo nahradiť národné jazyky, čo bolo...","Cielom nebolo nahradit narodne jazyky, co bolo..."
5,Hoci žiaden štát neprijal esperanto ako úradný...,Hoci ziaden stat neprijal esperanto ako uradny...
6,V Poľsku je na zozname nemateriálneho kultúrne...,V Polsku je na zozname nematerialneho kulturne...
7,"Získalo aj isté medzinárodné uznania, napríkla...","Ziskalo aj iste medzinarodne uznania, naprikla..."
8,V súčasnosti sa esperanto využíva pri cestovan...,V sucasnosti sa esperanto vyuziva pri cestovan...
9,Slovná zásoba esperanta pochádza predovšetkým ...,Slovna zasoba esperanta pochadza predovsetkym ...


In [5]:
# 5. Compute descriptive dataset statistics and export to JSON (stats.json)
all_text = " ".join(df['veta_s_diakritikou'])
diacritic_count = sum(1 for a, b in zip(all_text, " ".join(df['veta_bez_diakritiky'])) if a != b)
words = re.findall(r'\w+', all_text.lower())

stats = {
    "total_sentence_pairs": len(df),
    "train_pairs": len(train_df),
    "val_pairs": len(val_df),
    "avg_sentence_length_chars": round(len(all_text) / len(df), 1) if len(df) > 0 else 0,
    "total_characters": len(all_text),
    "diacritic_characters_count": diacritic_count,
    "unique_words": len(set(words))
}

with open('statistics.json', 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=4, ensure_ascii=False)

print("Štatistiky boli vygenerované:")
print(json.dumps(stats, indent=4, ensure_ascii=False))


Štatistiky boli vygenerované:
{
    "total_sentence_pairs": 20000,
    "train_pairs": 18000,
    "val_pairs": 2000,
    "avg_sentence_length_chars": 125.1,
    "total_characters": 2502504,
    "diacritic_characters_count": 183626,
    "unique_words": 66144
}


In [6]:
# 6. Export curated sentence dataset to Excel (vety_dataset.xlsx)
df.to_excel('vety_dataset.xlsx', index=False, engine='openpyxl')
files.download('vety_dataset.xlsx')
print("Súbor vety_dataset.xlsx je pripravený na stiahnutie.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Súbor vety_dataset.xlsx je pripravený na stiahnutie.
